In [ ]:
from pathlib import Path
import copy
import math
import os
import sys

import fpsample
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import seaborn as sns
import yaml
from matplotlib.colors import Normalize
from matplotlib.pyplot import rc_context
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from tqdm import tqdm

sys.path.insert(0, "../../../../../../shared_utils")
from experiment_utils import manual_filtering

## Initialize columns to keep 

In [ ]:
# Cell type fractions
celltype_fraction = {
    "CD14Mono": 0.02
}

columns_of_interest = [
    'gm-csf_[ng_ml]',
     'tpo_[ng_ml]',
     'sr1_[nm]',
     'um171_[nm]',
     'um729_[µm]',
     'scf_[ng_ml]',
     'butyzamide_[nm]',
     'retinoic_acid_[µm]',
     'ldl_[ng_ml]',
     'il3_[ng_ml]',
     'o2_[%]',
     'days_of_culture',
     'run_name',
     "cfg:constraints:c_scheduler_kwargs:t_warmup",
     "cfg:constraints:c_scheduler_kwargs:vmin",
     "cfg:constraints:c_scheduler_kwargs:vmax",

]

columns_of_interest_l4 =  columns_of_interest + ["il7_[ng_ml]",
                                                 "mcsf_[ng_ml]",
                                                 "ly_cocktail_[ul/well]", 
                                                 "loss", 
                                                 "target_ct_loss_std",
                                                 "target_ct_loss_mean",
                                                 "ct_prop_total_variance"]

## Load solutions 

In [ ]:
def load_pure_cell_type_solutions(
    path_to_results,
    paths,
    cell_types,
    load_uncertainties=True
):

    path_to_results = Path(path_to_results)

    pure_cell_type_solution_dict = {
        cell_type: [] for cell_type in cell_types
    }

    for experiment_folder in tqdm(paths):

        cell_type_file_paths = path_to_results / experiment_folder

        if not cell_type_file_paths.exists():
            continue

        # Iterate through cell types
        for cell_type in os.listdir(cell_type_file_paths):

            if cell_type not in cell_types:
                continue

            config_exp_folders = cell_type_file_paths / cell_type

            if not config_exp_folders.is_dir():
                continue

            # Iterate through experiment configs
            for config_exp_folder in os.listdir(config_exp_folders):

                exp_folder = config_exp_folders / config_exp_folder

                if not exp_folder.is_dir():
                    continue

                # Check candidates.csv exists
                if "candidates.csv" not in os.listdir(exp_folder):
                    continue

                if load_uncertainties:
                    uncertainty_path = (
                        exp_folder
                        / "uncertainty_annotation"
                        / "candidates_with_uncertainties.csv"
                    )

                else:
                    uncertainty_path = (
                        exp_folder
                        / "candidates.csv"
                    )

                # Check uncertainty annotation exists
                if uncertainty_path.exists():
 
                    candidate_with_uncertainty_df = pd.read_csv(
                        uncertainty_path
                    )

                    candidate_with_uncertainty_df["run_name"] = [
                        Path(file).name
                        for file in candidate_with_uncertainty_df[
                            "cfg:paths:dump_dir"
                        ]
                    ]

                    pure_cell_type_solution_dict[cell_type].append(
                        candidate_with_uncertainty_df
                    )

    return pure_cell_type_solution_dict

## Read annotation file

In [ ]:
# Read annotation files
ANNOTATION_CFG_FILE = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]
protocol_cols_l4 = protocol_cols + ['il7_[ng_ml]','mcsf_[ng_ml]','ly_cocktail_[ul/well]']

ANNOTATION_CFG_FILE_BOUNDS = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/constraints/default_all_cols_loop4.yaml"
with open(ANNOTATION_CFG_FILE_BOUNDS, "r") as fb:
    annotation_cfg_bound_l4 = yaml.safe_load(fb)
ubound_dict_l4, lbound_dict_l4 = annotation_cfg_bound_l4["ubound_dict"],  annotation_cfg_bound_l4["lbound_dict"]

bounds_l4 = {}
for param in lbound_dict_l4:
    bounds_l4[param] = (lbound_dict_l4[param], ubound_dict_l4[param])

### Collect optimized samples 

In [ ]:
path_to_results_loop4 = Path("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/loop4_replicate")
paths_loop4 = os.listdir(path_to_results_loop4)

In [ ]:
cell_types = ["CD14Mono"]

raw_solutions_loop4 = load_pure_cell_type_solutions(
    path_to_results_loop4,
    paths_loop4,
    cell_types,
    load_uncertainties=True
)

In [ ]:
raw_solutions_loop4 = {
    key: pd.concat(val, axis=0) for key, val in raw_solutions_loop4.items()
}

In [ ]:
cd14mono = raw_solutions_loop4['CD14Mono']

# Filter solutions

In [ ]:
# Collect filtered and annotated data frames 
filtered_df_cd14mono, annotated_df_cd14mono = manual_filtering(
    df_dict={"CD14Mono": cd14mono},
    bounds=bounds_l4,
    columns_to_keep=columns_of_interest_l4, 
    celltype_fraction=celltype_fraction,
    margin_frac_bounds=1, 
    protocol_cols=protocol_cols_l4,
    use_fraction_filter=True, 
    use_oxygen_days_filter=True,
    use_bounds_filter=False
)
cd14mono = filtered_df_cd14mono["CD14Mono"].reset_index()

In [ ]:
cd14mono = cd14mono[cd14mono["days_of_culture"] < 18]

## Check heatmaps

In [ ]:
cd14mono_prot = np.log1p(cd14mono.loc[:, protocol_cols_l4])
cd14mono_prop = cd14mono["CD14Mono_prop"]

In [ ]:
# Map the proportion values to colors using a continuous colormap
norm = Normalize(vmin=cd14mono_prop.min(), vmax=cd14mono_prop.max())
cmap = cm.viridis  # choose any colormap you like, e.g. "coolwarm", "magma"
row_colors = cd14mono_prop.map(lambda x: cmap(norm(x)))

g = sns.clustermap(
    cd14mono_prot,
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(5, 4),
)

# Add a separate colorbar legend for the row_colors annotation
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = g.fig.add_axes([1.02, 0.3, 0.03, 0.4])  # [left, bottom, width, height]
g.fig.colorbar(sm, cax=cbar_ax, label="late_MgkPro_prop")

plt.show()

In [ ]:
# 1. Get the linkage used for the row dendrogram
row_linkage = g.dendrogram_row.linkage

# 2. Cut the tree into k clusters (tune this by eye from the dendrogram)
n_clusters = 4
row_clusters = fcluster(row_linkage, t=n_clusters, criterion='maxclust')

# 3. Build a dataframe linking each row (sample) to its cluster + its proportion value
cluster_df = pd.DataFrame({
    'cluster': row_clusters,
    'cd14mono_prop': cd14mono_prop.values
}, index=cd14mono_prot.index)

# 4. Check which cluster(s) have the highest average proportion
cluster_summary = cluster_df.groupby('cluster')['cd14mono_prop'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(cluster_summary)

# 5. Pull out the actual samples in the top cluster(s)
top_cluster_id = cluster_summary.index[0]
high_prop_samples = cluster_df[cluster_df['cluster'] == top_cluster_id]
print(high_prop_samples)

Check if clusters are the ones we think

## Cluster 1

In [ ]:
cd14mono_cl1 = cd14mono.loc[cluster_df[cluster_df.cluster==1].index]

sns.clustermap(
    np.log1p(cd14mono_cl1.loc[:, protocol_cols_l4]),
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(5, 4),
)

In [ ]:
cd14mono_cl1_select = cd14mono_cl1.sort_values(by="target_ct_loss_std", ascending=True).iloc[:2]
cd14mono_cl1_select = pd.concat([cd14mono_cl1_select, cd14mono_cl1.iloc[[cd14mono_cl1["butyzamide_[nm]"].argmax()]]])
# cd14mono_cl1_select.to_csv("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered/cluster_analysis_loop4/cd16mono/cl1.csv")

In [ ]:
cd14mono_cl1_select.loc[:, protocol_cols_l4]

## Cluster 2

In [ ]:
cd14mono_cl2 = cd14mono.loc[cluster_df[cluster_df.cluster==2].index]

sns.clustermap(
    np.log1p(cd14mono_cl2.loc[:, protocol_cols_l4]),
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(5, 4),
)

In [ ]:
cd14mono_cl2_select = cd14mono_cl2.sort_values(by="target_ct_loss_std", ascending=True).iloc[:3]


cd14mono_cl2_select.loc[:, protocol_cols_l4]

In [ ]:
# cd14mono_cl2_select.to_csv("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered/cluster_analysis_loop4/cd16mono/cl2.csv")

## Cluster 3

In [ ]:
cd14mono_cl3 = cd14mono.loc[cluster_df[cluster_df.cluster==3].index]

sns.clustermap(
    np.log1p(cd14mono_cl3.loc[:, protocol_cols_l4]),
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(5, 4),
)

In [ ]:
cd14mono_cl3_select = cd14mono_cl3.sort_values(by="target_ct_loss_std", ascending=True).iloc[:3]
cd14mono_cl3_select.loc[:, protocol_cols_l4]

In [ ]:
# cd14mono_cl3_select.to_csv("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered/cluster_analysis_loop4/cd16mono/cl3.csv")

## Cluster 4

In [ ]:
cd14mono_cl4 = cd14mono.loc[cluster_df[cluster_df.cluster==4].index]

sns.clustermap(
    np.log1p(cd14mono_cl4.loc[:, protocol_cols_l4]),
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(5, 4),
)

In [ ]:
cd14mono_cl4_select = cd14mono_cl4.sort_values(by="target_ct_loss_std", ascending=True).iloc[:2]
cd14mono_cl4_select = pd.concat([cd14mono_cl4_select, cd14mono_cl4.iloc[[cd14mono_cl4["mcsf_[ng_ml]"].argmax()]]], axis=0)
cd14mono_cl4_select.loc[:, protocol_cols_l4]

In [ ]:
# cd14mono_cl4_select.to_csv("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered/cluster_analysis_loop4/cd16mono/cl4.csv")

## PCA with clusters

In [ ]:
with plt.rc_context({"figure.figsize": (3, 3), "figure.dpi": 150}):
    X_l4 = cd14mono_prot.values
    adata_l4 = sc.AnnData(X=X_l4.copy(), obs=cd14mono)
    adata_l4.obs["clusters"] = row_clusters
    adata_l4.obs["clusters"] = adata_l4.obs["clusters"].astype("category")

    sc.pp.pca(adata_l4)

    # Make sure the palette keys match the actual category dtype (str vs int)
    cluster_palette = {
        1: "#E63946",
        2: "#ea7f83",
        3: "#1D3557",
        4: "#376284",
        5: "#94C7CF",
    }
    cats = adata_l4.obs["clusters"].cat.categories
    if cats.dtype == object or all(isinstance(c, str) for c in cats):
        cluster_palette = {str(k): v for k, v in cluster_palette.items()}

    # Option A: keep all points, give every category a color
    all_colors = {c: "#CCCCCC" for c in cats}  # fallback grey for other clusters
    all_colors.update(cluster_palette)

    sc.pl.pca(
        adata_l4,
        s=200,
        color="clusters",
        frameon=False,
        groups=list(cluster_palette.keys()),
        palette=all_colors,
    )

    sc.pl.pca(adata_l4, s=200, color="CD14Mono_prop", frameon=False)